# 🧑‍⚕️ Exercícios — Sistemas Especialistas

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Construa um sistema especialista para diagnóstico médico simplificado, com interface de perguntas e tratamento de incerteza.


## 1. Sistema Especialista para Diagnóstico de Doenças

In [ ]:
# Base de conhecimento: regras de diagnóstico
base_conhecimento = {
    "gripe": {
        "sintomas": ["febre", "coriza", "tosse", "dor_corpo", "fadiga"],
        "necessarios": 3,  # mínimo de sintomas para o diagnóstico
        "tratamento": "Repouso, líquidos, antitérmico. Consulte um médico se piorar.",
        "urgencia": "baixa"
    },
    "covid": {
        "sintomas": ["febre", "tosse_seca", "perda_olfato", "perda_paladar", "falta_ar"],
        "necessarios": 3,
        "tratamento": "Isolamento, monitoramento, teste RT-PCR recomendado.",
        "urgencia": "media"
    },
    "alergia": {
        "sintomas": ["coriza", "espirros", "coceira_olhos", "lacrimejamento"],
        "necessarios": 2,
        "tratamento": "Anti-histamínico. Evite alérgenos.",
        "urgencia": "baixa"
    },
    "pneumonia": {
        "sintomas": ["febre_alta", "tosse_produtiva", "falta_ar", "dor_peito"],
        "necessarios": 3,
        "tratamento": "URGENTE: procure atendimento médico imediatamente.",
        "urgencia": "alta"
    },
}

def diagnosticar_sistema_especialista(sintomas_paciente):
    """Realiza diagnóstico baseado nos sintomas informados."""
    sintomas = set(sintomas_paciente)
    diagnosticos = []
    
    for doenca, info in base_conhecimento.items():
        sintomas_match = sintomas.intersection(info["sintomas"])
        if len(sintomas_match) >= info["necessarios"]:
            confianca = len(sintomas_match) / len(info["sintomas"])
            diagnosticos.append({
                "doenca": doenca,
                "sintomas_match": sintomas_match,
                "confianca": confianca,
                "tratamento": info["tratamento"],
                "urgencia": info["urgencia"]
            })
    
    diagnosticos.sort(key=lambda d: d["confianca"], reverse=True)
    return diagnosticos

# Teste
sintomas_caso1 = ["febre", "tosse", "coriza", "dor_corpo"]
sintomas_caso2 = ["febre", "tosse_seca", "perda_olfato", "falta_ar"]

for caso, sintomas in [("Caso 1", sintomas_caso1), ("Caso 2", sintomas_caso2)]:
    print(f"\n{'='*50}")
    print(f"{caso}: sintomas = {sintomas}")
    print('='*50)
    diags = diagnosticar_sistema_especialista(sintomas)
    if diags:
        for d in diags:
            print(f"  Diagnóstico: {d['doenca'].upper()}  (confiança: {d['confianca']:.0%})")
            print(f"  Sintomas compatíveis: {d['sintomas_match']}")
            print(f"  Urgência: {d['urgencia'].upper()}")
            print(f"  Tratamento: {d['tratamento']}\n")
    else:
        print("  Nenhum diagnóstico encontrado.")


### 📝 Exercício 1

Adicione **2 novas doenças** à base de conhecimento (ex: dengue, amigdalite). Crie um caso de teste para cada uma e execute o diagnóstico.

In [ ]:
base_expandida = dict(base_conhecimento)
# ✏️ Adicione novas doenças:
base_expandida["dengue"] = {
    # TODO: preencha sintomas, necessarios, tratamento, urgencia
}
# TODO: adicione mais uma doença

# Casos de teste:
novos_casos = [
    # TODO
]


## 2. Sistema com Encadeamento Progressivo e Certeza

In [ ]:
# Sistema especialista com fator de certeza (CF)
class RegraComCerteza:
    def __init__(self, condicoes, conclusao, cf):
        self.condicoes = condicoes  # lista de (fato, cf_minimo)
        self.conclusao = conclusao
        self.cf_regra = cf  # certeza da regra

class SistemaEspecialistaComCF:
    def __init__(self):
        self.fatos = {}  # fato → CF
        self.regras = []
    
    def adicionar_fato(self, fato, cf=1.0):
        self.fatos[fato] = cf
    
    def adicionar_regra(self, condicoes, conclusao, cf_regra):
        self.regras.append(RegraComCerteza(condicoes, conclusao, cf_regra))
    
    def inferir(self):
        mudou = True
        while mudou:
            mudou = False
            for regra in self.regras:
                if all(self.fatos.get(f, 0) >= cf_min
                       for f, cf_min in regra.condicoes):
                    cf_min_cond = min(self.fatos.get(f,0) for f,_ in regra.condicoes)
                    cf_novo = cf_min_cond * regra.cf_regra
                    if cf_novo > self.fatos.get(regra.conclusao, 0):
                        self.fatos[regra.conclusao] = cf_novo
                        mudou = True
    
    def consultar(self, fato):
        return self.fatos.get(fato, 0)
    
    def mostrar_resultados(self):
        print("\nResultados do sistema especialista:")
        for fato, cf in sorted(self.fatos.items(), key=lambda x: -x[1]):
            barra = "█" * int(cf*20)
            print(f"  {fato:<25} CF={cf:.2f}  |{barra:<20}|")

se = SistemaEspecialistaComCF()
# Fatos observados com CF
se.adicionar_fato("febre",      0.9)
se.adicionar_fato("tosse_seca", 0.8)
se.adicionar_fato("perda_olfato", 0.7)

# Regras com CF
se.adicionar_regra([("febre",0.5), ("tosse_seca",0.5)], "infeccao_viral",  0.8)
se.adicionar_regra([("infeccao_viral",0.6), ("perda_olfato",0.5)], "provavel_covid", 0.9)
se.adicionar_regra([("febre",0.5), ("tosse_seca",0.5)], "gripe",            0.6)

se.inferir()
se.mostrar_resultados()


### 📝 Exercício 2

Modifique os fatores de certeza dos fatos observados e observe como os diagnósticos mudam. O que acontece quando CF(febre) = 0.3?

In [ ]:
se2 = SistemaEspecialistaComCF()
se2.adicionar_fato("febre",      0.3)  # incerteza alta
se2.adicionar_fato("tosse_seca", 0.8)
se2.adicionar_fato("perda_olfato", 0.7)
se2.adicionar_regra([("febre",0.5), ("tosse_seca",0.5)], "infeccao_viral",  0.8)
se2.adicionar_regra([("infeccao_viral",0.6), ("perda_olfato",0.5)], "provavel_covid", 0.9)

se2.inferir()
se2.mostrar_resultados()
print("\nCF febre baixo → infecção viral?", se2.consultar("infeccao_viral"))


## 3. Exercício Final — Seu Próprio Sistema Especialista

Escolha um domínio de sua preferência (ex: diagnóstico de bugs de software, recomendação de carreira, triagem de suporte técnico) e crie um sistema especialista com pelo menos 5 regras.

In [ ]:
# ✏️ Implemente seu sistema especialista aqui:
meu_se = SistemaEspecialistaComCF()

# Domínio escolhido: _______________
# Fatos:
# meu_se.adicionar_fato("...", CF)

# Regras:
# meu_se.adicionar_regra([("...", CF_min)], "...", CF_regra)

meu_se.inferir()
meu_se.mostrar_resultados()
